# Piloto 2026 — validación de la observación del condensado

## tl;dr

**Veredicto:** `NO_VALID_MODEL`.  
**Modelo seleccionado:** `None`.

La prueba corrige una incompatibilidad de escala: los fermentadores contienen 230 L, mientras que 85–88 % de captura era una hipótesis nominal, no una recuperación medida. Se estima una fracción efectiva por compuesto, manteniendo fijos la producción biológica, el equilibrio gas/líquido y el balance de masa.

## Alcance y caveats

- La fracción ajustada representa todo el operador entre aroma emitido y masa recuperada: condensador, línea, trampa y pérdidas no observadas.
- No debe llamarse eficiencia termodinámica del condensador sin calibración con estándar gaseoso.
- El test fue motivado después de observar el fallo de los modelos de retardo y la escala de 230 L; es exploratorio.
- Leave-one-run-out mide transferencia interna retrospectiva. Aún se necesita una corrida prospectiva sellada.
- 26211-P-12 se conserva en el primario y sólo se excluye en sensibilidad.

In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display, Image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'fermentation_model').exists():
    ROOT = ROOT.parent
if not (ROOT / 'fermentation_model').exists():
    raise RuntimeError('Execute from the repository or a descendant directory')
sys.path.insert(0, str(ROOT / 'fermentation_model'))
from pilot_2026 import run_aroma_capture_efficiency_validation_2026 as analysis
result = analysis.load_results() if os.environ.get('PILOT_AROMA_REUSE_RESULTS') == '1' else analysis.run_analysis()
print('Veredicto:', result['gate']['verdict'])
print('Modelo:', result['gate']['selected_model'])
print('Eficiencias ajustadas:', result['summary']['fitted_capture_efficiency_all_data'])

## Escala y calidad de datos

In [ ]:
display(result['volume_context'])
display(result['anomaly_summary'].head(10).round(3))
display(Image(filename=analysis.FIGURE_DIR / '01_sample_anomaly_audit.png'))

## Validación leave-one-run-out

In [ ]:
display(result['metrics'].round(4))
display(result['comparison'].round(4))
display(Image(filename=analysis.FIGURE_DIR / '02_capture_efficiency_loro_nrmse.png'))
display(Image(filename=analysis.FIGURE_DIR / '07_capture_efficiency_by_fold.png'))

## Forzantes y curvas

In [ ]:
display(Image(filename=analysis.FIGURE_DIR / '03_rco2_temperature_pulse_drivers.png'))
for species in analysis.base.SPECIES_LABELS:
    display(Image(filename=analysis.FIGURE_DIR / f'04_liquid_{species}.png'))
    display(Image(filename=analysis.FIGURE_DIR / f'05_condensate_{species}.png'))

## Parámetros e identificabilidad

In [ ]:
display(result['stability'].round(5))
display(result['parameters'].round(6))
display(result['fit_validation'].round(5))

## Takeaways

- Si el modelo pasa, existe una corrección predictiva interna defendible, pero la fracción de captura sigue siendo efectiva y específica de este montaje.
- Si la fracción cambia fuertemente entre folds o cae en límites, la observación de condensado no es transferible con un único valor constante.
- El experimento confirmatorio debe medir simultáneamente concentración líquida, gas de salida y recuperación de trampa con un estándar, registrando volumen y %EtOH del condensado.

In [ ]:
assert result['gate']['verdict'] in {'PASS', 'NO_VALID_MODEL'}
assert len(result['figures']) == 8
assert result['fit_validation']['maximum_relative_mass_balance_error'].max() <= 1e-8
print('Notebook ejecutado sin errores; figuras:', len(result['figures']))